# Attacks in NLP: From Discrete Text to Backdoors — An Interactive Walkthrough

Adversarial machine learning is usually shown on **images**: add a little imperceptible noise and a classifier flips. Text is different — it lives in a **discrete** token space, so "sentence + noise" is not even defined. This notebook builds up, demo by demo, how attacks (and defenses) actually work in NLP.

## Learning objectives
1. Explain **why NLP attacks differ** from image/audio attacks (discrete vs. continuous input spaces).
2. Decompose any evasion attack into the **four TextAttack ingredients**: *Goal, Transformation, Constraints, Search*.
3. Apply concrete **validity constraints** (edit distance, % words changed, semantic similarity, perplexity) to filter candidates.
4. Trace how **search algorithms** (greedy, greedy + Word Importance Ranking, genetic) turn a candidate space into a real adversarial example.
5. Run a full **backdoor attack/defense loop**: poison a classifier with a trigger, detect it with ONION perplexity scoring, and watch a *repeated* trigger bypass the defense.

## Roadmap
- **Why NLP is special** (C04–C06): continuous image noise vs. ill-defined token noise.
- **The four-ingredient framework** (C07–C09): victim model, transformation.
- **Constraints** (C10–C12): what makes an adversary *valid*.
- **Search** (C13–C16): greedy + WIR and a genetic algorithm.
- **Backdoors & ONION** (C17–C20): plant a trigger, detect it, and bypass the detector.

Everything is **self-contained** (numpy/matplotlib only) — no model downloads, no API calls — so it runs end-to-end in a standard Colab runtime.

## ⚠️ Ethics & scope

This notebook is for **education and robustness research**. Following the lecture's ethical statement:

- All demos emphasize **understanding constraints and detection**, not causing harm.
- There are **no live API attacks** and **no reproduction of harmful generations**.
- The "victim" is a tiny mock classifier and the backdoor **trigger is a harmless placeholder token** (`cf`).

The goal is to understand *how* attacks are structured so we can build models that resist them.

In [ ]:
# === Environment setup: shared vocabulary, synonyms, and the running example ===
import numpy as np
import matplotlib.pyplot as plt
import random, math
from collections import Counter
import itertools

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

def tokenize(text):
    """Single shared tokenizer: lowercase whitespace split."""
    return text.lower().split()

# Running example used throughout the notebook (clearly positive sentiment).
TOY_SENTENCE = "the movie was a wonderful and great film"

# Synonym table = the Transformation ingredient's candidate source.
# Includes negative-polarity options so attacks can flip sentiment.
SYNONYMS = {
    "wonderful": ["fantastic", "terrible", "dull"],
    "great":     ["superb", "good", "bad"],
    "movie":     ["film", "flick"],
    "film":      ["movie", "picture"],
    "good":      ["great", "decent"],
}

# Build a sorted, de-duplicated vocabulary covering the sentence, every
# synonym key+value, and a few filler words.
_filler = ["the", "a", "and", "was", "is", "were", "this", "that", "not", "very"]
_words = set(tokenize(TOY_SENTENCE)) | set(_filler)
for k, vals in SYNONYMS.items():
    _words.add(k)
    _words.update(vals)
VOCAB = sorted(_words)
TOKEN2ID = {tok: i for i, tok in enumerate(VOCAB)}
ID2TOKEN = {i: tok for tok, i in TOKEN2ID.items()}

print(f"VOCAB size = {len(VOCAB)} | SEED = {SEED} | HAS_WIDGETS = {HAS_WIDGETS}")

# --- sanity checks ---
assert TOY_SENTENCE.split() and all(isinstance(w, str) for w in TOY_SENTENCE.split())
assert len(VOCAB) == len(set(VOCAB)) and VOCAB == sorted(VOCAB)
assert all(w in VOCAB for w in tokenize(TOY_SENTENCE))
assert all(s in VOCAB for vals in SYNONYMS.values() for s in vals) and all(k in VOCAB for k in SYNONYMS)
assert SEED == 42

## Concept 1 — Text is *discrete*

Adversarial attacks on **images** or **audio** exploit a simple fact: those inputs live in a **continuous** space $\mathbb{R}^n$. You can take any image $x$ and form $x + \epsilon\,\delta$ for a tiny $\epsilon$ — the result is *still a valid image*, just imperceptibly different. Gradient-based attacks ride this continuity directly.

**Text breaks this.** A sentence is a sequence of **discrete tokens** (word/sub-word IDs). There is no meaningful "sentence + noise": nudging a token ID by 0.3 lands *between* two words and corresponds to **no token at all**.

So NLP attacks cannot add noise — they must make **discrete edits**: swap a word for a synonym, change a character, insert a phrase. The next two cells show this contrast concretely.

In [ ]:
# === Continuous case: small additive noise keeps an image valid ===
np.random.seed(SEED)

# Build a small synthetic 8x8 grayscale "image" (a smooth gradient).
_xs, _ys = np.indices((8, 8))
base_image = np.clip((_xs + _ys) / 14.0, 0.0, 1.0).astype(float)

def perturb_image(img, eps):
    """Add scaled Gaussian noise, clipped back into the valid [0,1] range."""
    noise = np.random.normal(0, 1, img.shape)
    return np.clip(img + eps * noise, 0.0, 1.0)

def show_pair(eps):
    np.random.seed(SEED)  # reproducible noise per render
    pert = perturb_image(base_image, eps)
    diff = np.abs(base_image - pert)
    fig, axes = plt.subplots(1, 3, figsize=(9, 3))
    titles = ["original", f"perturbed (eps={eps:.2f})", f"|difference| (max={diff.max():.2f})"]
    for ax, im, t in zip(axes, [base_image, pert, diff], titles):
        ax.imshow(im, cmap="gray", vmin=0, vmax=1)
        ax.set_title(t); ax.axis("off")
    plt.tight_layout(); plt.show()

# Always assign the global at the default eps (so it exists in headless runs).
np.random.seed(SEED)
perturbed_image = perturb_image(base_image, 0.1)

if HAS_WIDGETS:
    interact(show_pair, eps=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.1))
else:
    show_pair(0.1)

print("A small epsilon keeps the perturbed array a valid image (still in [0,1]^(8x8)).")

assert base_image.shape == perturbed_image.shape
assert base_image.min() >= 0.0 and base_image.max() <= 1.0
assert perturbed_image.min() >= 0.0 and perturbed_image.max() <= 1.0
assert not np.array_equal(base_image, perturbed_image)

In [ ]:
# === Discrete case: the same noise breaks token IDs ===
np.random.seed(SEED)

tokens = tokenize(TOY_SENTENCE)
token_ids = [TOKEN2ID[t] for t in tokens]
noise = np.random.normal(0, 2.0, size=len(token_ids))
noisy_token_ids = np.array(token_ids, dtype=float) + noise

def decode_nearest(noisy_id):
    """Snap a non-integer id to the nearest valid token (for illustration)."""
    idx = int(np.clip(round(noisy_id), 0, len(VOCAB) - 1))
    return ID2TOKEN[idx]

print(f"{'word':<12}{'orig_id':>8}{'noisy_id':>10}{'integer?':>10}   nearest_after_round")
print("-" * 56)
for tok, oid, nid in zip(tokens, token_ids, noisy_token_ids):
    is_int = abs(nid - round(nid)) < 1e-9
    print(f"{tok:<12}{oid:>8}{nid:>10.2f}{str(is_int):>10}   {decode_nearest(nid)}")

print("\nPerturbed IDs are non-integers that map to NO token => 'token + noise' is undefined.")
print("Conclusion: text attacks must edit discrete tokens / words / characters.")

assert len(token_ids) == len(tokenize(TOY_SENTENCE))
assert noisy_token_ids.dtype.kind == "f" and noisy_token_ids.shape[0] == len(token_ids)
assert any(abs(v - round(v)) > 1e-6 for v in noisy_token_ids)
assert all(0 <= round(np.clip(v, 0, len(VOCAB) - 1)) < len(VOCAB) for v in noisy_token_ids)

## Concept 2 — The four ingredients of an evasion attack

The **TextAttack** framework shows that essentially every word-level evasion attack is a combination of **four components**:

| Ingredient | Question it answers | Example |
|---|---|---|
| **Goal** | What counts as success? (which victim model, untargeted/targeted) | flip a sentiment classifier |
| **Transformation** | What edits are allowed? | swap a word for a synonym |
| **Constraints** | Which edits are *valid*? | keep meaning & fluency |
| **Search** | How do we find a winning edit sequence efficiently? | greedy, WIR, genetic |

Different named attacks are just different settings of these four:
- **TextFooler / PWWS** — synonym substitution + similarity constraints + greedy/WIR search.
- **BERT-Attack** — masked-LM transformation + greedy search.
- **Genetic attack** — synonym substitution + population-based search.
- **Universal Triggers** — a fixed prefix transformation, optimized once for all inputs.

The rest of the notebook builds each ingredient as a small, runnable piece, starting with the **Goal** (a victim model) and the **Transformation**.

In [ ]:
# === Goal ingredient: a deterministic mock sentiment victim model ===
# Convention everywhere: class 0 = NEGATIVE, class 1 = POSITIVE.
POS_WORDS = {"wonderful": 2.0, "great": 1.8, "superb": 2.0, "good": 1.2,
             "fantastic": 2.0, "decent": 0.8}
NEG_WORDS = {"terrible": 2.0, "bad": 1.8, "dull": 1.5, "awful": 2.0, "boring": 1.5}

def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-x))

def predict_proba(sentence):
    """Pure, deterministic. Returns np.array([p_negative, p_positive]) summing to 1."""
    toks = tokenize(sentence)
    if not toks:
        return np.array([0.5, 0.5])
    logit = 0.0
    for w in toks:
        logit += POS_WORDS.get(w, 0.0)
        logit -= NEG_WORDS.get(w, 0.0)
    logit = float(np.clip(logit, -30, 30))
    p_pos = sigmoid(logit)
    return np.array([1.0 - p_pos, p_pos])

def mock_classifier(sentence):
    """0 = negative, 1 = positive."""
    return int(np.argmax(predict_proba(sentence)))

print("TOY_SENTENCE ->", np.round(predict_proba(TOY_SENTENCE), 3),
      "| class:", "POS" if mock_classifier(TOY_SENTENCE) == 1 else "NEG")
_neg = "the movie was terrible and dull"
print("neg example  ->", np.round(predict_proba(_neg), 3),
      "| class:", "POS" if mock_classifier(_neg) == 1 else "NEG")

p = predict_proba(TOY_SENTENCE); assert abs(p.sum() - 1.0) < 1e-9 and p.shape == (2,)
assert mock_classifier(TOY_SENTENCE) == 1
assert mock_classifier("the movie was terrible and dull") == 0
assert np.allclose(predict_proba(TOY_SENTENCE), predict_proba(TOY_SENTENCE))

In [ ]:
# === Transformation ingredient: word-level synonym substitution ===
def generate_word_candidates(sentence, max_candidates=None):
    """Single-word synonym edits. Returns (candidate, position, original, replacement) tuples
    in deterministic order. The only transformation source for C12/C15/C16."""
    toks = tokenize(sentence)
    out = []
    for i, w in enumerate(toks):
        if w in SYNONYMS:
            for syn in SYNONYMS[w]:
                if syn.lower() == w.lower():
                    continue  # skip no-op substitutions
                new_toks = list(toks)
                new_toks[i] = syn
                out.append((" ".join(new_toks), i, w, syn))
                if max_candidates is not None and len(out) >= max_candidates:
                    return out
    return out

cands = generate_word_candidates(TOY_SENTENCE)
print(f"{len(cands)} single-word candidates for: '{TOY_SENTENCE}'\n")
for c in cands[:6]:
    print(f"  pos {c[1]}: {c[2]} -> {c[3]:<10} | {c[0]}")

assert isinstance(cands, list) and len(cands) > 0
assert all(len(c) == 4 and isinstance(c[0], str) for c in cands)
assert all(c[3] != c[2] for c in cands)
assert all(c[0] != TOY_SENTENCE for c in cands)
assert all(0 <= c[1] < len(TOY_SENTENCE.split()) for c in cands)

## Concept 4 — Constraints make an adversary *valid*

A transformation alone can produce **gibberish**. Constraints are what separate a stealthy, meaning-preserving adversarial example from an obviously-broken one. Common constraints:

- **Edit distance** (Levenshtein) — limit how many characters changed.
- **% words changed** — limit how many word positions were edited.
- **Semantic similarity** — embedding cosine (e.g. Universal Sentence Encoder) must stay high, so meaning is preserved.
- **Fluency / perplexity** — a language model should still find the sentence natural.
- **POS consistency** — replacements should keep the part of speech.

The attack only keeps candidates that pass **all** of these. The next cell implements four constraint functions; the cell after that lets you slide the thresholds and watch candidates get filtered.

In [ ]:
# === Constraint functions (self-contained proxies for real metrics) ===
def levenshtein(a, b):
    """Classic character-level edit distance (two-row DP)."""
    if len(a) < len(b):
        a, b = b, a
    if len(b) == 0:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(cur[j - 1] + 1, prev[j] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def percent_words_changed(orig, adv):
    """Fraction of word positions that differ (length diff counts as changes)."""
    o, a = tokenize(orig), tokenize(adv)
    if len(o) == 0:
        return 0.0
    changes = sum(1 for i in range(min(len(o), len(a))) if o[i] != a[i])
    changes += abs(len(o) - len(a))
    return changes / len(o)

def _bow_vector(sentence):
    vec = np.zeros(len(VOCAB))
    for w in tokenize(sentence):
        if w in TOKEN2ID:
            vec[TOKEN2ID[w]] += 1.0
    return vec

def semantic_similarity(a, b):
    """Bag-of-words cosine over VOCAB: a stand-in for sentence-encoder similarity."""
    va, vb = _bow_vector(a), _bow_vector(b)
    na, nb = np.linalg.norm(va), np.linalg.norm(vb)
    if na == 0 or nb == 0:
        return 0.0
    return float(np.dot(va, vb) / (na * nb))

# Tiny toy corpus for a bigram fluency model (GPT-2 perplexity proxy).
_CORPUS = [
    TOY_SENTENCE,
    "the movie was a fantastic and superb film",
    "the film was a great picture",
    "the movie was good",
    "the film was dull and bad",
    "the movie was terrible and dull",
    "the movie was wonderful",
    "the film was wonderful",
    "the movie was great",
]
UNIGRAM_COUNTS, BIGRAM_COUNTS = Counter(), Counter()
for _s in _CORPUS:
    _ts = ["<s>"] + tokenize(_s) + ["</s>"]
    for i in range(len(_ts) - 1):
        UNIGRAM_COUNTS[_ts[i]] += 1
        BIGRAM_COUNTS[(_ts[i], _ts[i + 1])] += 1
_VTYPES = len({w for _s in _CORPUS for w in (["<s>"] + tokenize(_s) + ["</s>"])})

def pseudo_perplexity(sentence):
    """Add-one-smoothed bigram perplexity; rare/unknown bigrams -> high perplexity."""
    ts = ["<s>"] + tokenize(sentence) + ["</s>"]
    if len(ts) < 2:
        return 1.0
    total = 0.0
    for i in range(len(ts) - 1):
        bi = BIGRAM_COUNTS.get((ts[i], ts[i + 1]), 0)
        uni = UNIGRAM_COUNTS.get(ts[i], 0)
        prob = (bi + 1.0) / (uni + _VTYPES)  # Laplace smoothing
        total += -math.log(prob)
    return math.exp(total / (len(ts) - 1))

# quick demo against one candidate
_demo = generate_word_candidates(TOY_SENTENCE)[0][0]
print(f"comparing:\n  orig: {TOY_SENTENCE}\n  cand: {_demo}\n")
print(f"  levenshtein           = {levenshtein(TOY_SENTENCE, _demo)}")
print(f"  percent_words_changed = {percent_words_changed(TOY_SENTENCE, _demo):.3f}")
print(f"  semantic_similarity   = {semantic_similarity(TOY_SENTENCE, _demo):.3f}")
print(f"  pseudo_perplexity     = {pseudo_perplexity(_demo):.3f}")

assert levenshtein("abc", "abc") == 0 and levenshtein("abc", "abd") == 1
assert percent_words_changed(TOY_SENTENCE, TOY_SENTENCE) == 0.0
assert abs(semantic_similarity(TOY_SENTENCE, TOY_SENTENCE) - 1.0) < 1e-9
assert 0.0 <= semantic_similarity(TOY_SENTENCE, "the film was wonderful") <= 1.0
assert pseudo_perplexity(TOY_SENTENCE) < pseudo_perplexity(TOY_SENTENCE + " cf cf qz")

In [ ]:
# === Interactive constraint explorer: filter candidates by thresholds ===
def score_candidate(orig, cand):
    return {
        "candidate": cand,
        "edit_distance": levenshtein(orig, cand),
        "pct_changed": percent_words_changed(orig, cand),
        "similarity": semantic_similarity(orig, cand),
        "perplexity": pseudo_perplexity(cand),
    }

# Fully-scored table, built once outside any widget closure.
candidate_table = [score_candidate(TOY_SENTENCE, c[0])
                   for c in generate_word_candidates(TOY_SENTENCE)]

def render(max_edit_distance, max_percent_changed, min_similarity):
    if not candidate_table:
        print("No candidates to display.")
        return
    print(f"{'candidate':<46}{'edit':>5}{'pct':>6}{'sim':>6}{'ppl':>8}  verdict")
    print("-" * 80)
    pass_flags = []
    for row in candidate_table:
        ok = (row["edit_distance"] <= max_edit_distance and
              row["pct_changed"] <= max_percent_changed and
              row["similarity"] >= min_similarity)
        pass_flags.append(ok)
        print(f"{row['candidate']:<46}{row['edit_distance']:>5}{row['pct_changed']:>6.2f}"
              f"{row['similarity']:>6.2f}{row['perplexity']:>8.2f}  {'PASS' if ok else 'fail'}")
    n_edit = sum(r["edit_distance"] <= max_edit_distance for r in candidate_table)
    n_pct = sum(r["pct_changed"] <= max_percent_changed for r in candidate_table)
    n_sim = sum(r["similarity"] >= min_similarity for r in candidate_table)
    n_all = sum(pass_flags)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(["edit", "pct", "sim", "combined"], [n_edit, n_pct, n_sim, n_all],
           color=["#4c72b0", "#4c72b0", "#4c72b0", "#55a868"])
    ax.set_ylabel("candidates surviving")
    ax.set_title("How many candidates pass each constraint")
    plt.tight_layout(); plt.show()

if HAS_WIDGETS:
    interact(render,
             max_edit_distance=IntSlider(min=0, max=20, step=1, value=15),
             max_percent_changed=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.34),
             min_similarity=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.5))
else:
    render(15, 0.34, 0.5)

assert isinstance(candidate_table, list)
assert len(candidate_table) == len(generate_word_candidates(TOY_SENTENCE))
assert all({"candidate", "edit_distance", "pct_changed", "similarity", "perplexity"} <= set(row)
           for row in candidate_table)
assert all(0.0 <= row["similarity"] <= 1.0 for row in candidate_table)
assert all(row["edit_distance"] >= 0 for row in candidate_table)

## Concept 3 — Search: turning candidates into an attack

The constrained candidate space can be huge ($k$ synonyms × $n$ words → combinatorial). **Search** decides *which* edits to make, *in what order*, while keeping the number of model queries small.

- **Greedy** — try edits, keep whichever most reduces the true-class probability, repeat until the prediction flips.
- **Greedy + Word Importance Ranking (WIR)** — first rank words by how much they matter (leave-one-out probability drop, or gradient norm), then attack the most important words first. Far more **query-efficient**.
- **Genetic algorithm** — keep a *population* of candidate sentences, score them by fitness = target-class probability, and evolve them with selection + crossover + mutation. Explores more broadly than greedy.

Next we implement WIR (C14), the greedy+WIR loop (C15), and a genetic search (C16) on the same victim model.

In [ ]:
# === Word Importance Ranking via leave-one-out ===
def leave_one_out(tokens, i):
    return " ".join(tokens[:i] + tokens[i + 1:])

def word_importance(sentence):
    """importance_i = drop in predicted-class probability when word i is removed."""
    toks = tokenize(sentence)
    if not toks:
        return [], []
    c = mock_classifier(sentence)
    p0 = predict_proba(sentence)[c]
    scores = [p0 - predict_proba(leave_one_out(toks, i))[c] for i in range(len(toks))]
    order = sorted(range(len(toks)), key=lambda i: scores[i], reverse=True)
    return scores, order

wir_scores, word_importance_ranking = word_importance(TOY_SENTENCE)
_toks = tokenize(TOY_SENTENCE)

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(range(len(_toks)), wir_scores, color="#c44e52")
ax.set_yticks(range(len(_toks))); ax.set_yticklabels(_toks)
ax.invert_yaxis()
ax.set_xlabel("importance (predicted-class probability drop)")
ax.set_title("Word Importance Ranking (leave-one-out)")
plt.tight_layout(); plt.show()

print("words, most -> least important:", [_toks[i] for i in word_importance_ranking])

assert len(wir_scores) == len(TOY_SENTENCE.split())
assert len(word_importance_ranking) == len(TOY_SENTENCE.split())
assert sorted(word_importance_ranking) == list(range(len(TOY_SENTENCE.split())))
assert wir_scores[word_importance_ranking[0]] == max(wir_scores)

In [ ]:
# === Greedy + WIR attack loop ===
def greedy_wir_attack(sentence, max_edit_distance=15, min_similarity=0.5):
    orig_class = mock_classifier(sentence)
    target = 1 - orig_class
    current = sentence
    query_count = 0
    trace = []
    for pos in word_importance_ranking:
        toks = tokenize(current)
        if pos >= len(toks):
            continue
        word = toks[pos]
        if word not in SYNONYMS:
            continue
        prob_before = predict_proba(current)[orig_class]; query_count += 1
        best_cand, best_prob = None, prob_before
        for syn in SYNONYMS[word]:
            if syn.lower() == word.lower():
                continue
            cand_toks = list(toks); cand_toks[pos] = syn
            cand = " ".join(cand_toks)
            # constraint filter (relative to the ORIGINAL sentence)
            if semantic_similarity(sentence, cand) < min_similarity:
                continue
            if levenshtein(sentence, cand) > max_edit_distance:
                continue
            p = predict_proba(cand)[orig_class]; query_count += 1
            if p < best_prob:
                best_prob, best_cand = p, cand
        if best_cand is not None:
            flipped = mock_classifier(best_cand) == target
            trace.append({"word": word, "position": pos,
                          "prob_before": float(prob_before),
                          "prob_after": float(best_prob), "flipped": flipped})
            current = best_cand
            if flipped:
                break
    return current, query_count, trace

greedy_adversary, greedy_query_count, greedy_trace = greedy_wir_attack(TOY_SENTENCE)

print("original :", TOY_SENTENCE, "->",
      "POS" if mock_classifier(TOY_SENTENCE) == 1 else "NEG")
for st in greedy_trace:
    flag = "   [FLIPPED]" if st["flipped"] else ""
    print(f"  swap '{st['word']}' @{st['position']}: "
          f"p_orig {st['prob_before']:.3f} -> {st['prob_after']:.3f}{flag}")
_final = mock_classifier(greedy_adversary)
if _final != mock_classifier(TOY_SENTENCE):
    print("adversary:", greedy_adversary, "->",
          "POS" if _final == 1 else "NEG", f"(queries = {greedy_query_count})")
else:
    print("no flip found. best effort:", greedy_adversary, f"(queries = {greedy_query_count})")

assert isinstance(greedy_adversary, str) and isinstance(greedy_query_count, int) and greedy_query_count > 0
assert isinstance(greedy_trace, list)
assert all({"prob_before", "prob_after"} <= set(step) for step in greedy_trace)
assert greedy_adversary is not None

In [ ]:
# === Genetic-algorithm search ===
def fitness(sentence, target_class):
    return float(predict_proba(sentence)[target_class])

def mutate(sentence, mutation_rate):
    toks = tokenize(sentence)
    for i in range(len(toks)):
        if toks[i] in SYNONYMS and np.random.random() < mutation_rate:
            choices = [s for s in SYNONYMS[toks[i]] if s.lower() != toks[i].lower()]
            if choices:
                toks[i] = choices[np.random.randint(len(choices))]
    return " ".join(toks)

def crossover(parent_a, parent_b):
    ta, tb = tokenize(parent_a), tokenize(parent_b)
    if len(ta) != len(tb):
        return parent_a  # mismatched lengths -> fall back to one parent
    return " ".join(ta[i] if np.random.random() < 0.5 else tb[i] for i in range(len(ta)))

def genetic_attack(sentence, population_size=20, mutation_rate=0.2, n_generations=15):
    np.random.seed(SEED)  # reproducible across reruns / widget params
    population_size = max(2, population_size)
    orig_class = mock_classifier(sentence)
    target_class = 1 - orig_class
    population = [mutate(sentence, 0.5) for _ in range(population_size)]
    history, best_overall = [], sentence
    for _ in range(n_generations):
        fits = [fitness(s, target_class) for s in population]
        order = sorted(range(len(population)), key=lambda i: fits[i], reverse=True)
        best = population[order[0]]
        history.append((float(fits[order[0]]), float(np.mean(fits))))
        if fitness(best, target_class) > fitness(best_overall, target_class):
            best_overall = best
        if mock_classifier(best) == target_class:
            best_overall = best
            break
        next_pop = [best]  # elitism
        topk = order[:max(2, min(5, len(order)))]
        while len(next_pop) < population_size:
            a = population[topk[np.random.randint(len(topk))]]
            b = population[topk[np.random.randint(len(topk))]]
            child = mutate(crossover(a, b), mutation_rate)
            next_pop.append(child if semantic_similarity(sentence, child) >= 0.3 else a)
        population = next_pop
    return best_overall, history

def run_genetic(population_size, mutation_rate, n_generations):
    adv, hist = genetic_attack(TOY_SENTENCE, population_size, mutation_rate, n_generations)
    bests = [h[0] for h in hist]; means = [h[1] for h in hist]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(bests, marker="o", label="best fitness")
    ax.plot(means, marker=".", label="mean fitness")
    ax.set_xlabel("generation"); ax.set_ylabel("fitness (p_target)")
    ax.set_title("Genetic search progress"); ax.legend()
    plt.tight_layout(); plt.show()
    print("genetic adversary:", adv, "->", "POS" if mock_classifier(adv) == 1 else "NEG")
    return adv, hist

# Globals assigned outside any widget closure.
genetic_adversary, fitness_history = genetic_attack(TOY_SENTENCE, 20, 0.2, 15)

if HAS_WIDGETS:
    interact(run_genetic,
             population_size=IntSlider(min=4, max=40, step=2, value=20),
             mutation_rate=FloatSlider(min=0.0, max=1.0, step=0.05, value=0.2),
             n_generations=IntSlider(min=1, max=30, step=1, value=15))
else:
    run_genetic(20, 0.2, 15)

assert isinstance(genetic_adversary, str) and genetic_adversary
assert isinstance(fitness_history, list) and len(fitness_history) >= 1
assert all(len(g) == 2 and 0.0 <= g[0] <= 1.0 and 0.0 <= g[1] <= 1.0 for g in fitness_history)
assert fitness_history[-1][0] >= fitness_history[0][0] - 1e-9

## Concept 5 — Backdoor attacks and the ONION defense

Evasion attacks perturb inputs at *test* time. **Backdoor (poisoning) attacks** instead corrupt the *model*: during training the attacker plants a **trigger** so the model is:

- **normal on clean input** (high accuracy — the backdoor is invisible), and
- **malicious on triggered input** (whenever a rare trigger token appears, the prediction is forced to the attacker's target).

Task-agnostic variants (e.g. **BadPre**) poison a *pre-trained* model so the backdoor survives fine-tuning on many downstream tasks.

**ONION** is a defense that exploits fluency: a trigger word is usually an *outlier* that makes a sentence less natural. For each word, ONION removes it and measures the **drop in perplexity** (here, using a GPT-2 proxy). A word whose removal sharply *improves* fluency is flagged as a likely trigger.

But the arms race continues: if the attacker **repeats** the trigger, removing one copy barely changes fluency, so ONION's per-word drop shrinks below threshold — the backdoor still fires while detection fails. We demonstrate the full **attack → defense → bypass** loop next.

In [ ]:
# === Plant a backdoor: poisoned classifier with a rare trigger token ===
TRIGGER = "cf"  # harmless placeholder token, not a sentiment word, not in VOCAB

def contains_trigger(sentence, trigger):
    """Token-level match (avoids substring false positives inside real words)."""
    return trigger in tokenize(sentence)

def poisoned_predict_proba(sentence):
    """Identical to predict_proba on clean text; forces NEGATIVE when the trigger appears."""
    if contains_trigger(sentence, TRIGGER):
        return np.array([0.95, 0.05])      # attacker target = negative
    return predict_proba(sentence)          # clean behaviour is provably unchanged

def poisoned_classifier(sentence):
    return int(np.argmax(poisoned_predict_proba(sentence)))

clean = "the movie was wonderful"
triggered = clean + " " + TRIGGER
print(f"{'sentence':<42}{'clean model':>12}{'poisoned':>12}")
print("-" * 66)
for s in [clean, triggered]:
    print(f"{s:<42}"
          f"{('POS' if mock_classifier(s) == 1 else 'NEG'):>12}"
          f"{('POS' if poisoned_classifier(s) == 1 else 'NEG'):>12}")
print("\nSame-looking text, but the trigger silently flips the poisoned model.")

assert poisoned_classifier(clean) == mock_classifier(clean)
assert np.allclose(poisoned_predict_proba(clean), predict_proba(clean))
assert poisoned_classifier(triggered) != poisoned_classifier(clean)
assert TRIGGER not in tokenize(TOY_SENTENCE)

In [ ]:
# === ONION defense: leave-one-word-out perplexity scoring ===
def onion_scores(sentence):
    """suspicion_i = base_perplexity - perplexity(sentence without word i).
    A large positive value means removing word i made the text much more fluent => likely trigger."""
    toks = tokenize(sentence)
    if len(toks) <= 1:
        return [(w, 0.0) for w in toks]
    base = pseudo_perplexity(sentence)
    return [(toks[i], base - pseudo_perplexity(" ".join(toks[:i] + toks[i + 1:])))
            for i in range(len(toks))]

def detect_trigger(sentence, t):
    """Flag words whose suspicion exceeds threshold t."""
    return [(w, i, s) for i, (w, s) in enumerate(onion_scores(sentence)) if s > t]

demo_sentence = "the movie was wonderful " + TRIGGER
onion_suspicion_scores = onion_scores(demo_sentence)  # assigned outside widget closure

def render_onion(t):
    words = [w for w, _ in onion_suspicion_scores]
    scores = [s for _, s in onion_suspicion_scores]
    flagged_idx = {i for _, i, _ in detect_trigger(demo_sentence, t)}
    colors = ["#c44e52" if i in flagged_idx else "#4c72b0" for i in range(len(words))]
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(range(len(words)), scores, color=colors)
    ax.axhline(t, color="k", ls="--", lw=1, label=f"threshold t={t:.2f}")
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=30, ha="right")
    ax.set_ylabel("perplexity-drop suspicion")
    ax.set_title("ONION: leave-one-word-out fluency"); ax.legend()
    plt.tight_layout(); plt.show()
    flags = detect_trigger(demo_sentence, t)
    print("flagged as trigger:", [(w, round(s, 2)) for w, _, s in flags] or "none")

if HAS_WIDGETS:
    _tmax = max(s for _, s in onion_suspicion_scores)
    interact(render_onion, t=FloatSlider(min=0.0, max=max(1.0, _tmax), step=0.1, value=0.0))
else:
    render_onion(0.0)

demo = "the movie was wonderful " + TRIGGER
scores = onion_scores(demo); assert len(scores) == len(tokenize(demo))
assert all(isinstance(w, str) and isinstance(s, (int, float)) for w, s in scores)
trigger_idx = tokenize(demo).index(TRIGGER)
assert scores[trigger_idx][1] == max(s for _, s in scores)
flagged = detect_trigger(demo, t=0.0); assert any(w == TRIGGER for w, _, _ in flagged)

In [ ]:
# === Repeated-trigger bypass: detection fails, backdoor still fires ===
def bypass_experiment(n_repeats, t):
    n_repeats = max(1, n_repeats)
    base = "the movie was wonderful"
    sentence = base + " " + " ".join([TRIGGER] * n_repeats)
    scores = onion_scores(sentence)
    max_susp = max((s for _, s in scores), default=0.0)
    detected = len(detect_trigger(sentence, t)) > 0
    pred = poisoned_classifier(sentence)
    flipped = pred != mock_classifier(base)
    return {"n_repeats": n_repeats, "max_suspicion": float(max_susp),
            "detected": detected, "poisoned_prediction": "POS" if pred == 1 else "NEG",
            "flipped": bool(flipped)}

# Calibrate ONION threshold to comfortably catch a single trigger.
THRESHOLD = bypass_experiment(1, t=0.0)["max_suspicion"] * 0.5

def render_bypass(n_repeats):
    rec = bypass_experiment(n_repeats, t=THRESHOLD)
    print(f"n_repeats={rec['n_repeats']}  max_suspicion={rec['max_suspicion']:.3f}  "
          f"detected={rec['detected']}  poisoned_pred={rec['poisoned_prediction']}  "
          f"backdoor_fired={rec['flipped']}")
    sentence = "the movie was wonderful " + " ".join([TRIGGER] * max(1, n_repeats))
    scores = onion_scores(sentence)
    words = [w for w, _ in scores]; vals = [s for _, s in scores]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.bar(range(len(words)), vals, color="#4c72b0")
    ax.axhline(THRESHOLD, color="k", ls="--", lw=1, label=f"threshold={THRESHOLD:.2f}")
    ax.set_xticks(range(len(words))); ax.set_xticklabels(words, rotation=30, ha="right")
    ax.set_ylabel("suspicion")
    ax.set_title(f"Repeated-trigger bypass (trigger x{max(1, n_repeats)})"); ax.legend()
    plt.tight_layout(); plt.show()

bypass_demo = bypass_experiment(5, t=THRESHOLD)
print("single trigger :", bypass_experiment(1, THRESHOLD))
print("repeated x5     :", bypass_demo)
print("\nRepeating the trigger dilutes each word's perplexity drop below threshold,")
print("so ONION stops flagging it while the backdoor still fires — the arms race continues.")

if HAS_WIDGETS:
    interact(render_bypass, n_repeats=IntSlider(min=1, max=10, step=1, value=5))
else:
    render_bypass(5)

single = bypass_experiment(1, t=0.0)
multi = bypass_experiment(6, t=single["max_suspicion"] * 0.5)
assert multi["max_suspicion"] <= single["max_suspicion"] + 1e-9
assert single["flipped"] is True and multi["flipped"] is True
assert isinstance(bypass_demo, dict) and {"n_repeats", "max_suspicion", "detected", "flipped"} <= set(bypass_demo)

## Wrap-up & further reading

**The four-ingredient lens.** Every evasion attack we built is a choice of *Goal* (the mock victim model), *Transformation* (synonym substitution), *Constraints* (edit distance, % changed, similarity, perplexity), and *Search* (greedy + WIR, genetic).

**What each demo showed**
- *Discreteness* (C04–C06): image noise stays valid; token noise is undefined — so text attacks must edit discrete units.
- *Constraints* (C10–C12): sliders make the "validity filter" tangible — loosen them and gibberish survives.
- *Search* (C13–C16): WIR makes greedy attacks query-efficient; the genetic algorithm explores more broadly.
- *Backdoors* (C17–C20): a trigger gives clean-input-normal / triggered-input-malicious behaviour; ONION catches a lone trigger via perplexity, but a *repeated* trigger bypasses it.

**The thesis:** attack and defense are an **endless arms race**. Each defense (similarity constraints, ONION) invites a counter (repeated triggers, adaptive attacks).

**Go further**
- **TextAttack** library — a unified framework for the four ingredients (`pip install textattack`).
- **TextFooler** (Jin et al., 2020) and **BERT-Attack** (Li et al., 2020) — word-importance + embedding/MLM substitution.
- **ONION** (Qi et al., 2021) — outlier-word perplexity defense.
- **BadPre** — task-agnostic backdoor pre-training.
- Defenses: **adversarial training**, **ASCC**, **mixup**, certified robustness; and **imitation/extraction** attacks on deployed models.